# CAMUS — DRL Boundary Refinement

One notebook, three agents. Switch which agent runs by changing the YAML path in Cell 1:

| Agent | Config | Action space |
|---|---|---|
| **DDQN**   | `configs/camus_dqn.yaml`     | 7 discrete |
| **Dueling** | `configs/camus_dueling.yaml` | 7 discrete |
| **DDPG**   | `configs/camus_ddpg.yaml`    | 2D continuous |

Also pick which **structure** to refine via `target_class` in the YAML (1=LV_endo, 2=LV_epi, 3=LA). One run = one structure.

## Kaggle setup
1. Attach: **CAMUS dataset** + **iteris-pkg** + **camus-baseline-outputs** (the trained U-Net checkpoint)
2. Accelerator: **GPU T4 x2**, Persistence: **Files only**, Internet: **On**
3. Save Version → Save & Run All (Commit)

## 0 · Install + locate package

In [ ]:
import subprocess, sys
from pathlib import Path

init_files = list(Path('/kaggle/input').rglob('iteris/__init__.py'))
if not init_files:
    raise RuntimeError('iteris-pkg not attached.')
PKG_ROOT = init_files[0].parent.parent
REQ      = PKG_ROOT / 'requirements.txt'
subprocess.run(['pip', 'install', '-r', str(REQ), '--quiet', '--upgrade'], check=True)
sys.path.insert(0, str(PKG_ROOT))
print(f'✓ Setup complete. Package at: {PKG_ROOT}')

## 1 · Load DRL config + baseline config
Edit `CFG_NAME` to pick the agent.

In [ ]:
CFG_NAME = 'camus_dqn.yaml'      # ← switch to camus_dueling.yaml or camus_ddpg.yaml

from iteris.config import load_config, load_drl_config
from iteris.utils  import get_device

cfg          = load_drl_config(str(PKG_ROOT / 'configs' / CFG_NAME))
baseline_cfg = load_config(    str(PKG_ROOT / 'configs' / cfg['baseline_cfg_name']))

# Locate CAMUS data root + baseline checkpoint regardless of mount path.
camus_roots = [p for p in Path('/kaggle/input').rglob('CAMUS') if p.is_dir()]
if camus_roots:
    baseline_cfg['data_root'] = str(camus_roots[0])

ckpt_cands = [p for p in Path('/kaggle/input').rglob('camus_best.pt')]
if ckpt_cands:
    cfg['baseline_checkpoint'] = str(ckpt_cands[0])

cfg['checkpoint_dir'] = '/kaggle/working'

print(f'Agent           : {cfg["agent_type"]}')
print(f'Target class    : {cfg["target_class"]} ({baseline_cfg["class_names"][cfg["target_class"]]})')
print(f'CAMUS data root : {baseline_cfg["data_root"]}')
print(f'Baseline ckpt   : {cfg["baseline_checkpoint"]}')
print(f'Train steps     : {cfg["train_steps"]}')
print(f'Device          : {get_device()}')

## 2 · Warm-start — pre-compute U-Net init masks
Runs the trained U-Net over the whole dataset once and caches predictions.
Avoids 50k+ wasted forward passes during DRL training.

In [ ]:
from iteris.warm_start import precompute_init_masks

train_samples, val_samples, test_samples = precompute_init_masks(
    baseline_cfg        = baseline_cfg,
    baseline_checkpoint = cfg['baseline_checkpoint'],
    target_class        = cfg['target_class'],
    min_area_fraction   = cfg.get('min_area_fraction', 0.01),
)
print(f'\nSamples — train: {len(train_samples)}  val: {len(val_samples)}  test: {len(test_samples)}')
import numpy as np
init_dices = [float((s['init_mask'] & s['gt_mask']).sum() * 2 /
                    (s['init_mask'].sum() + s['gt_mask'].sum() + 1e-6))
              for s in val_samples]
print(f'U-Net init Dice on val: mean {np.mean(init_dices):.4f}  median {np.median(init_dices):.4f}')

## 3 · Run DRL training

In [ ]:
from iteris.drl_training import run_drl_training

result    = run_drl_training(cfg, train_samples, val_samples)
agent     = result['agent']
history   = result['history']
best_dice = result['best_dice']

print(f'\n✓ Training complete. Best val final-Dice: {best_dice:.4f}')
print(f'  Checkpoint: {result["checkpoint"]}')

## 4 · Learning curves

In [ ]:
import matplotlib.pyplot as plt

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(13, 4))
ax1.plot(history['step'], history['init_dice_mean'],  label='Init Dice (U-Net)', linestyle='--', alpha=0.6)
ax1.plot(history['step'], history['final_dice_mean'], label=f'Final Dice ({cfg["agent_type"]})')
ax1.set(xlabel='Training step', ylabel='Mean Val Dice', title='Refinement learning curve')
ax1.legend(); ax1.grid(alpha=0.3)

ax2.plot(history['step'], history['delta_dice_mean'], color='C2')
ax2.axhline(0, color='k', lw=0.5)
ax2.set(xlabel='Training step', ylabel='Δ Dice (final − init)',
        title=f'Refinement gain ({cfg["agent_type"]} c{cfg["target_class"]})')
ax2.grid(alpha=0.3)

plt.suptitle(f'CAMUS {cfg["agent_type"]} — class {cfg["target_class"]} ({baseline_cfg["class_names"][cfg["target_class"]]})')
plt.tight_layout()
plt.savefig(f'/kaggle/working/{cfg["agent_type"].lower()}_c{cfg["target_class"]}_curves.png', dpi=150)
plt.show()

## 5 · Test-set evaluation

In [ ]:
from iteris.drl_training import evaluate_agent

test_metrics = evaluate_agent(agent, test_samples, env_kwargs={
    'action_type': agent.action_type,
    'max_steps':   cfg.get('max_steps', 20),
    'shift_px':    cfg.get('shift_px', 2),
    'sdt_clip':    cfg.get('sdt_clip', 20.0),
    'reward_clip': cfg.get('reward_clip', 1.0),
    'cont_action_scale': cfg.get('cont_action_scale', 0.04),
})
import json
print(json.dumps(test_metrics, indent=2))

with open(f'/kaggle/working/{cfg["agent_type"].lower()}_c{cfg["target_class"]}_test_results.json', 'w') as f:
    json.dump({**test_metrics, 'agent_type': cfg['agent_type'], 'target_class': cfg['target_class'], 'best_val_dice': best_dice}, f, indent=2)
print('Saved test results JSON.')